In [2]:
using System;
using System.Text;
using System.Collections.Generic;

// ====================== Вспомогательные функции ======================
static string[] CanvasToString(char[,] canvas) {
    var result = new string[canvas.GetLength(0)];
    for (int i = 0; i < canvas.GetLength(0); i++) {
        var sb = new StringBuilder();
        for (int j = 0; j < canvas.GetLength(1); j++)
            sb.Append(canvas[i, j]);
        result[i] = sb.ToString();
    }
    return result;
}

static void PrintStrings(string[] lines) {
    foreach (var line in lines) Console.WriteLine(line);
}

// ====================== Прямоугольник ======================
string[] GetRectangle(int width, int height, char symbol = '*', bool filled = false, int borderThickness = 1) {
    if (width <= 0 || height <= 0 || borderThickness <= 0) return new string[0];
    var canvas = new char[height, width];
    for (int i = 0; i < height; i++)
        for (int j = 0; j < width; j++)
            canvas[i, j] = ' ';
    int maxThick = Math.Min(width, height) / 2;
    borderThickness = Math.Min(borderThickness, maxThick);
    for (int i = 0; i < height; i++) {
        for (int j = 0; j < width; j++) {
            bool inBorder = i < borderThickness || i >= height - borderThickness ||
                            j < borderThickness || j >= width - borderThickness;
            if (filled || inBorder) canvas[i, j] = symbol;
        }
    }
    return CanvasToString(canvas);
}

void PrintRectangle(int width, int height, char symbol = '*', bool filled = false, int borderThickness = 1) =>
    PrintStrings(GetRectangle(width, height, symbol, filled, borderThickness));

// ====================== Равнобедренный прямоугольный треугольник ======================
string[] GetRightTriangle(int size, char symbol = '*', bool filled = false, string orientation = "bottom-left") {
    if (size <= 0) return new string[0];
    var canvas = new char[size, size];
    for (int i = 0; i < size; i++)
        for (int j = 0; j < size; j++)
            canvas[i, j] = ' ';
    for (int i = 0; i < size; i++) {
        for (int j = 0; j < size; j++) {
            bool draw = orientation switch {
                "bottom-left" => j <= i,
                "bottom-right" => j >= size - 1 - i,
                "top-left" => i + j <= size - 1,
                "top-right" => i <= j,
                _ => j <= i
            };
            if (filled ? draw : draw && (j == 0 || i == size - 1 || (orientation switch {
                "bottom-left" => j == i,
                "bottom-right" => j == size - 1 - i,
                "top-left" => i + j == size - 1,
                "top-right" => i == j,
                _ => false
            }))) canvas[i, j] = symbol;
        }
    }
    return CanvasToString(canvas);
}

// ====================== Песочные часы ======================
string[] GetHourglass(int size, char symbol = '*') {
    if (size <= 1 || size % 2 == 0) size++; // только нечётный размер для симметрии
    var canvas = new char[size, size];
    for (int i = 0; i < size; i++)
        for (int j = 0; j < size; j++)
            canvas[i, j] = ' ';
    int mid = size / 2;
    for (int i = 0; i < size; i++) {
        int offset = i <= mid ? i : size - 1 - i;
        for (int j = offset; j < size - offset; j++)
            canvas[i, j] = symbol;
    }
    return CanvasToString(canvas);
}

// ====================== Ромб ======================
string[] GetDiamond(int size, char symbol = '*', bool filled = false) {
    if (size <= 1) size = 3;
    if (size % 2 == 0) size++;
    var canvas = new char[size, size];
    for (int i = 0; i < size; i++)
        for (int j = 0; j < size; j++)
            canvas[i, j] = ' ';
    int mid = size / 2;
    for (int i = 0; i < size; i++) {
        int offset = Math.Abs(mid - i);
        for (int j = offset; j < size - offset; j++) {
            if (filled || j == offset || j == size - offset - 1)
                canvas[i, j] = symbol;
        }
    }
    return CanvasToString(canvas);
}

// ====================== Зебра (горизонтальные полосы) ======================
string[] GetZebra(int width, int height, int stripeThickness, char symbol1 = '#', char symbol2 = ' ') {
    if (width <= 0 || height <= 0 || stripeThickness <= 0) return new string[0];
    var canvas = new char[height, width];
    for (int i = 0; i < height; i++) {
        char current = (i / stripeThickness) % 2 == 0 ? symbol1 : symbol2;
        for (int j = 0; j < width; j++)
            canvas[i, j] = current;
    }
    return CanvasToString(canvas);
}

// ====================== Змейка (заполнение зигзагом) ======================
string[] GetSnake(int rows, int cols, bool clockwise = true, string direction = "horizontal") {
    if (rows <= 0 || cols <= 0) return new string[0];
    var canvas = new char[rows, cols];
    for (int i = 0; i < rows; i++)
        for (int j = 0; j < cols; j++)
            canvas[i, j] = ' ';
    if (direction == "horizontal") {
        for (int i = 0; i < rows; i++) {
            if (i % 2 == 0) {
                for (int j = 0; j < cols; j++) canvas[i, j] = '*';
            } else {
                for (int j = cols - 1; j >= 0; j--) canvas[i, j] = '*';
            }
        }
    } else { // vertical
        for (int j = 0; j < cols; j++) {
            if (j % 2 == 0) {
                for (int i = 0; i < rows; i++) canvas[i, j] = '*';
            } else {
                for (int i = rows - 1; i >= 0; i--) canvas[i, j] = '*';
            }
        }
    }
    return CanvasToString(canvas);
}

// ====================== Вложенные прямоугольники ======================
string[] GetNestedRectangles(int count, int spacing = 1, char symbol = '*') {
    if (count <= 0) return new string[0];
    int size = 1 + 2 * spacing * (count - 1);
    var canvas = new char[size, size];
    for (int i = 0; i < size; i++)
        for (int j = 0; j < size; j++)
            canvas[i, j] = ' ';
    for (int k = 0; k < count; k++) {
        int offset = k * spacing;
        int end = size - 1 - offset;
        for (int i = offset; i <= end; i++) {
            canvas[i, offset] = symbol;
            canvas[i, end] = symbol;
            canvas[offset, i] = symbol;
            canvas[end, i] = symbol;
        }
    }
    return CanvasToString(canvas);
}

// ====================== Улитка (спираль) ======================
string[] GetSnail(int size, bool clockwise = true) {
    if (size <= 1) size = 3;
    if (size % 2 == 0) size++;
    var canvas = new char[size, size];
    for (int i = 0; i < size; i++)
        for (int j = 0; j < size; j++)
            canvas[i, j] = ' ';
    int x = size / 2, y = size / 2;
    int dx = clockwise ? 1 : 0, dy = clockwise ? 0 : -1;
    int steps = 1, stepCount = 0, turnCount = 0;
    canvas[y, x] = '*';
    while (true) {
        for (int s = 0; s < steps; s++) {
            x += dx; y += dy;
            if (x < 0 || x >= size || y < 0 || y >= size) break;
            canvas[y, x] = '*';
        }
        if (x < 0 || x >= size || y < 0 || y >= size) break;
        // поворот
        if (clockwise) {
            (dx, dy) = (-dy, dx);
        } else {
            (dx, dy) = (dy, -dx);
        }
        turnCount++;
        if (turnCount % 2 == 0) steps++;
    }
    return CanvasToString(canvas);
}


(176,20): warning CS0219: Переменной "stepCount" присвоено значение, но оно ни разу не использовано.



In [3]:
Console.WriteLine("=== Прямоугольник с рамкой толщиной 2, не залитый ===");
PrintRectangle(10, 5, '#', false, 2);

Console.WriteLine("\n=== Прямоугольник залитый ===");
PrintRectangle(8, 4, '@', true);

Console.WriteLine("\n=== Прямоугольный треугольник (bottom-left) ===");
PrintStrings(GetRightTriangle(7, '*', false, "bottom-left"));

Console.WriteLine("\n=== Прямоугольный треугольник (top-right, залитый) ===");
PrintStrings(GetRightTriangle(7, '#', true, "top-right"));

Console.WriteLine("\n=== Песочные часы ===");
PrintStrings(GetHourglass(9, 'O'));

Console.WriteLine("\n=== Ромб (контур) ===");
PrintStrings(GetDiamond(9, '*', false));

Console.WriteLine("\n=== Ромб (залитый) ===");
PrintStrings(GetDiamond(7, '#', true));

Console.WriteLine("\n=== Зебра (толщина 2) ===");
PrintStrings(GetZebra(20, 8, 2, '#', '.'));

Console.WriteLine("\n=== Змейка горизонтальная ===");
PrintStrings(GetSnake(5, 10, true, "horizontal"));

Console.WriteLine("\n=== Змейка вертикальная ===");
PrintStrings(GetSnake(7, 7, true, "vertical"));

Console.WriteLine("\n=== Вложенные прямоугольники (3 шт., отступ 2) ===");
PrintStrings(GetNestedRectangles(3, 2, '+'));

Console.WriteLine("\n=== Улитка (по часовой) ===");
PrintStrings(GetSnail(9, true));

Console.WriteLine("\n=== Улитка (против часовой) ===");
PrintStrings(GetSnail(9, false));

=== Прямоугольник с рамкой толщиной 2, не залитый ===
##########
##########
##      ##
##########
##########

=== Прямоугольник залитый ===
@@@@@@@@
@@@@@@@@
@@@@@@@@
@@@@@@@@

=== Прямоугольный треугольник (bottom-left) ===
*      
**     
* *    
*  *   
*   *  
*    * 
*******

=== Прямоугольный треугольник (top-right, залитый) ===
#######
 ######
  #####
   ####
    ###
     ##
      #

=== Песочные часы ===
OOOOOOOOO
 OOOOOOO 
  OOOOO  
   OOO   
    O    
   OOO   
  OOOOO  
 OOOOOOO 
OOOOOOOOO

=== Ромб (контур) ===
    *    
   * *   
  *   *  
 *     * 
*       *
 *     * 
  *   *  
   * *   
    *    

=== Ромб (залитый) ===
   #   
  ###  
 ##### 
#######
 ##### 
  ###  
   #   

=== Зебра (толщина 2) ===
####################
####################
....................
....................
####################
####################
....................
....................

=== Змейка горизонтальная ===
**********
**********
**********
**********
**********

=== Змейка вертикал